In [75]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
import os
import random
import copy

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(0)
random.seed(0)
print(f"Using device: {DEVICE}")

# Load BBC News Summary data
base_dir = "BBC News Summary"
articles_dir = os.path.join(base_dir, "News Articles")
summaries_dir = os.path.join(base_dir, "Summaries")

pairs = []
max_article_len = 40
max_summary_len = 10

categories = os.listdir(articles_dir)
for cat in categories:
    cat_article_dir = os.path.join(articles_dir, cat)
    cat_summary_dir = os.path.join(summaries_dir, cat)
    if not os.path.isdir(cat_article_dir):
        continue
    for fname in sorted(os.listdir(cat_article_dir)):
        if not fname.endswith(".txt"):
            continue
        article_path = os.path.join(cat_article_dir, fname)
        summary_path = os.path.join(cat_summary_dir, fname)
        if not os.path.exists(summary_path):
            continue
        with open(article_path, "r", encoding="latin-1") as f:
            lines = f.read().strip().split("\n")
        article = " ".join(lines[2:]) if len(lines) > 2 else lines[0]
        article_words = article.split()[:max_article_len]
        if len(article_words) < 5:
            continue
        with open(summary_path, "r", encoding="latin-1") as f:
            summary = f.read().strip()
        summary_words = summary.split()[:max_summary_len]
        if len(summary_words) < 3:
            continue
        pairs.append((" ".join(article_words).lower(), " ".join(summary_words).lower()))

print(f"Loaded {len(pairs)} article-summary pairs")

# Use the full dataset
random.shuffle(pairs)
train_split = int(0.7 * len(pairs))
val_split = int(0.85 * len(pairs))
train_pairs = pairs[:train_split]
val_pairs = pairs[train_split:val_split]
test_pairs = pairs[val_split:]
print(f"Training pairs: {len(train_pairs)}, Validation pairs: {len(val_pairs)}, Test pairs: {len(test_pairs)}")

Using device: cpu
Loaded 2225 article-summary pairs
Training pairs: 1557, Validation pairs: 334, Test pairs: 334


In [76]:
# Build vocabulary
SPECIAL = ["<PAD>", "<SOS>", "<EOS>", "<UNK>"]
words = set()
for a, b in pairs:
    words.update(a.split())
    words.update(b.split())
vocab = SPECIAL + sorted(words)
stoi = {w: i for i, w in enumerate(vocab)}
itos = {i: w for w, i in stoi.items()}
vocab_size = len(vocab)
print(f"Vocabulary size: {vocab_size}")

def encode(text, add_sos=False, add_eos=False):
    ids = []
    if add_sos:
        ids.append(stoi["<SOS>"])
    for w in text.split():
        ids.append(stoi.get(w, stoi["<UNK>"]))
    if add_eos:
        ids.append(stoi["<EOS>"])
    return torch.tensor(ids, dtype=torch.long)

train_data = [(encode(a), encode(b, True, True)) for a, b in train_pairs]
val_data = [(encode(a), encode(b, True, True)) for a, b in val_pairs]
test_data = [(encode(a), encode(b, True, True)) for a, b in test_pairs]

Vocabulary size: 16081


In [77]:
EMBEDDING_SIZE = 128
HIDDEN_SIZE = 256
VOCAB_SIZE = vocab_size
TARGET_LEN = max_summary_len + 2
BATCH_SIZE = 16

In [78]:
class SummarizationDataset(Dataset):
    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]

def collate_fn(batch):
    srcs, tgts = zip(*batch)
    src_lengths = torch.tensor([len(s) for s in srcs], dtype=torch.long)
    src_padded = pad_sequence(srcs, batch_first=True, padding_value=stoi["<PAD>"])

    tgt_padded = torch.full((len(tgts), TARGET_LEN), stoi["<PAD>"], dtype=torch.long)
    for i, t in enumerate(tgts):
        tgt_padded[i, :len(t)] = t[:TARGET_LEN]

    return src_padded, src_lengths, tgt_padded

train_loader = DataLoader(SummarizationDataset(train_data), batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(SummarizationDataset(val_data), batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

In [79]:
shared_embedding = nn.Embedding(VOCAB_SIZE, EMBEDDING_SIZE)

In [80]:
class Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = shared_embedding
        self.rnn = nn.GRU(EMBEDDING_SIZE, HIDDEN_SIZE, batch_first=True)

    def forward(self, x, lengths):
        e = self.embedding(x)
        packed = nn.utils.rnn.pack_padded_sequence(e, lengths.cpu(), batch_first=True, enforce_sorted=False)
        packed_outputs, hidden = self.rnn(packed)
        outputs, _ = nn.utils.rnn.pad_packed_sequence(packed_outputs, batch_first=True, total_length=x.size(1))
        return outputs, hidden

In [81]:
class BahdanauAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.W_s = nn.Linear(HIDDEN_SIZE, HIDDEN_SIZE)
        self.W_h = nn.Linear(HIDDEN_SIZE, HIDDEN_SIZE)
        self.v = nn.Linear(HIDDEN_SIZE, 1)

    def forward(self, decoder_hidden, encoder_outputs, mask=None):
        query = decoder_hidden.unsqueeze(1)
        energy = torch.tanh(self.W_s(query) + self.W_h(encoder_outputs))  # (batch, seq_len, hidden)
        scores = self.v(energy).squeeze(-1)  # (batch, seq_len)

        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))

        attn_weights = F.softmax(scores, dim=1)  # (batch, seq_len)
        context = torch.bmm(attn_weights.unsqueeze(1), encoder_outputs)
        context = context.squeeze(1)  # (batch, hidden)
        return context, attn_weights

In [82]:
class Decoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = shared_embedding
        self.attention = BahdanauAttention()
        self.rnn = nn.GRU(EMBEDDING_SIZE+HIDDEN_SIZE, HIDDEN_SIZE, batch_first=True)
        self.fc_out = nn.Linear(HIDDEN_SIZE*2+EMBEDDING_SIZE, VOCAB_SIZE)


    def forward_step(self, input_token, hidden, encoder_outputs, mask=None):
        embedded = self.embedding(input_token.unsqueeze(1))
        context, attn_weights = self.attention(hidden.squeeze(0), encoder_outputs, mask)
        rnn_input = torch.cat((embedded, context.unsqueeze(1)), dim=2)
        output, hidden = self.rnn(rnn_input, hidden)
        output = output.squeeze(1)  # (batch, hidden)

        pred_input = torch.cat((output, context, embedded.squeeze(1)), dim=1)
        prediction = self.fc_out(pred_input)
        return prediction, hidden, attn_weights



    def forward(self, tgt, encoder_last_hidden, encoder_outputs, mask=None, teacher_forcing_ratio=1.0):
        batch_size = tgt.size(0)
        input_token = tgt[:, 0]
        hidden = encoder_last_hidden

        outputs = torch.zeros(batch_size, TARGET_LEN, VOCAB_SIZE, device=tgt.device)

        for t in range(1, TARGET_LEN):
            # print(itos[input_token.item()])
            prediction, hidden, attn_weights = self.forward_step(input_token, hidden, encoder_outputs, mask)
            outputs[:, t] = prediction

            use_teacher_forcing = random.random() < teacher_forcing_ratio
            input_token = tgt[:, t] if use_teacher_forcing else prediction.argmax(1)

        return outputs, attn_weights

In [83]:
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device

    def forward(self, src, src_lengths, tgt, teacher_forcing_ratio=1.0):
        encoder_outputs, encoder_hidden = self.encoder(src, src_lengths)
        mask = (src != stoi["<PAD>"]).to(self.device)
        outputs, attn_weights = self.decoder(tgt, encoder_hidden, encoder_outputs, mask, teacher_forcing_ratio)
        return outputs, attn_weights


    @torch.no_grad()
    def greedy_decode(self, src, sos_idx, eos_idx, max_len=TARGET_LEN):
        self.eval()
        src_lengths = torch.tensor([src.size(1)], dtype=torch.long)
        encoder_outputs, hidden = self.encoder(src, src_lengths)
        input_token = torch.tensor([sos_idx], device=self.device)

        tokens, attn_matrix = [], []
        for _ in range(max_len):
            prediction, hidden, attn_weights = self.decoder.forward_step(input_token, hidden, encoder_outputs)
            next_token = prediction.argmax(1)
            tokens.append(next_token.item())
            attn_matrix.append(attn_weights.squeeze(0).tolist())
            if next_token.item() == eos_idx:
                break
            input_token = next_token

        return tokens, attn_matrix

In [84]:
encoder = Encoder().to(DEVICE)
decoder = Decoder().to(DEVICE)
model = Seq2Seq(encoder, decoder, DEVICE).to(DEVICE)

optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss(ignore_index=stoi["<PAD>"])

In [87]:
def evaluate(loader):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for src_batch, src_lengths, tgt_batch in loader:
            src_batch = src_batch.to(DEVICE)
            tgt_batch = tgt_batch.to(DEVICE)
            outputs, _ = model(src_batch, src_lengths, tgt_batch)
            loss = criterion(outputs.permute(0, 2, 1), tgt_batch)
            total_loss += loss.item()
    return total_loss / len(loader)

PATIENCE = 5
best_val_loss = float("inf")
epochs_no_improve = 0
best_model_state = None

for epoch in range(100):
    model.train()
    total_loss = 0
    for src_batch, src_lengths, tgt_batch in train_loader:
        src_batch = src_batch.to(DEVICE)
        tgt_batch = tgt_batch.to(DEVICE)

        optimizer.zero_grad()
        outputs, _ = model(src_batch, src_lengths, tgt_batch)
        loss = criterion(outputs.permute(0, 2, 1), tgt_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    train_loss = total_loss / len(train_loader)
    val_loss = evaluate(val_loader)
    print(f"Epoch {epoch+1}, Loss: {train_loss}, Val Loss: {val_loss}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        epochs_no_improve = 0
        best_model_state = copy.deepcopy(model.state_dict())
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= PATIENCE:
            print(f"Early stopping at epoch {epoch+1} (no improvement for {PATIENCE} epochs)")
            break

if best_model_state is not None:
    model.load_state_dict(best_model_state)


Epoch 1, Loss: 4.492351945565671, Val Loss: 7.019446986062186
Epoch 2, Loss: 3.6590689347714793, Val Loss: 7.175105594453358
Epoch 3, Loss: 2.9515029216299253, Val Loss: 7.340556303660075
Epoch 4, Loss: 2.386591548822364, Val Loss: 7.412769817170643
Epoch 5, Loss: 1.9415358402291123, Val Loss: 7.523224035898845
Epoch 6, Loss: 1.6005714304593144, Val Loss: 7.680811927432106
Early stopping at epoch 6 (no improvement for 5 epochs)


In [86]:
model.eval()
for src, tgt in test_data:
    src = src.unsqueeze(0).to(DEVICE)
    tgt = tgt.unsqueeze(0).to(DEVICE)
    predicted_tokens, attn_matrix = model.greedy_decode(src, stoi["<SOS>"], stoi["<EOS>"])
    predicted_summary = " ".join([itos[idx] for idx in predicted_tokens if idx not in (stoi["<SOS>"], stoi["<EOS>"], stoi["<PAD>"])])
    actual_summary = " ".join([itos[idx.item()] for idx in tgt[0] if idx.item() not in (stoi["<SOS>"], stoi["<EOS>"], stoi["<PAD>"])])
    actual_article = " ".join([itos[idx.item()] for idx in src[0] if idx.item() not in (stoi["<SOS>"], stoi["<EOS>"], stoi["<PAD>"])])
    print(f"Article: {actual_article}")
    print(f"Actual Summary: {actual_summary}")
    print(f"Predicted: {predicted_summary}")
    print("=" * 50)


Article: wales coach mike ruddock says england lock danny grewcock needs to review his actions after he kicked dwayne peel. trouble flared at a ruck in the first half of wales' 11-9 win in cardiff as grewcock came recklessly over the
Actual Summary: wales coach mike ruddock says england lock danny grewcock needs
Predicted: the report predicts that the most difficult in the
Article: next has said its annual profit will be â£5m lower than previously expected because its end-of-year clearance sale has proved disappointing. "clearance rates in our end-of-season sale have been below our expectations," the company said. the high street retailer said
Actual Summary: next chief executive simon wolfson admitted that festive sales were
Predicted: the government has said the government to be of the
Article: uk sportswear firm umbro has posted a 222% rise in annual profit after sales of replica england football kits were boosted by the euro 2004 tournament. pre-tax profit for 2004 was â£15.4m ($29